# Нагрузочное тестирование vLLM (2xT4)

### 1. Установка зависимостей
Устанавливаем `vllm` и другие нужные библиотеки.

In [1]:
!pip install -q vllm protobuf==3.20.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 3.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 106.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 109.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━

### 2. Инициализация инференс-движка

In [2]:
import time
from vllm import LLM, SamplingParams
import torch
import pandas as pd

# модель AWQ (сжатая)
MODEL_NAME = "casperhansen/deepseek-r1-distill-qwen-14b-awq"

print("Инициализация vLLM Engine...")

# инициализация vLLM
# tensor_parallel_size=2 задействует обе T4
# gpu_memory_utilization=0.9 выделяет место под KV-кэш
llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",          
    tensor_parallel_size=2, 
    gpu_memory_utilization=0.90, 
    max_model_len=8192,
    trust_remote_code=True,
    enforce_eager=True,
    attention_backend="TRITON_ATTN"
)

print("Engine готов.")

# --- ТЕСТ 1: Latency для одного пользователя ---
prompts_single = ["Реши уравнение: x^2 - 5x + 6 = 0 по шагам"]
sampling_params = SamplingParams(max_tokens=450)

start_t = time.time()
outputs_single = llm.generate(prompts_single, sampling_params)
end_t = time.time()

generated_toks = len(outputs_single[0].outputs[0].token_ids)
single_speed = generated_toks / (end_t - start_t)
itl_ms = (1 / single_speed) * 1000

print(f"\nМетрики для одного пользователя:")
print(f"Скорость генерации: {single_speed:.2f} ток/сек")
print(f"Задержка между токенами (ITL): {itl_ms:.2f} мс")


# --- ТЕСТ 2: Throughput (батч из 15 пользователей) ---
# Создаем нагрузку из 15 запросов
prompts_batch =[
    "Объясни простыми словами, что такое квантовая запутанность.",
    "Вычисли значение выражения: 25 * 25 - 10.",
    "Напиши код на Python для алгоритма быстрой сортировки (quicksort).",
    "Какой город является столицей Франции?",
    "Подробно объясни процесс фотосинтеза.",
    "Выведи знаменитую формулу E=mc^2 шаг за шагом.",
    "Кто такой Исаак Ньютон и в чем его главные заслуги?",
    "Переведи слово 'Привет' на испанский язык.",
    "Назови 5 полезных продуктов питания для улучшения работы мозга.",
    "Как работает двигатель внутреннего сгорания?",
    "Реши квадратное уравнение: x^2 + 5x + 6 = 0.",
    "Напиши короткое стихотворение о космосе.",
    "Как работает сетевой протокол TCP/IP?",
    "Что такое черная дыра с точки зрения физики?",
    "Вычисли площадь круга, если его радиус равен 5."
]

print(f"\nЗапуск нагрузочного теста ({len(prompts_batch)} одновременных запросов)...")

start_batch = time.time()
outputs_batch = llm.generate(prompts_batch, sampling_params)
end_batch = time.time()

# Считаем общую пропускную способность
total_tokens_batch = sum([len(o.outputs[0].token_ids) for o in outputs_batch])
total_time_batch = end_batch - start_batch
system_throughput = total_tokens_batch / total_time_batch

print(f"\nМетрики под нагрузкой (Батч = 15):")
print(f"Всего сгенерировано токенов: {total_tokens_batch}")
print(f"Общее время генерации: {total_time_batch:.2f} сек")
print(f"Общая пропускная способность (Throughput): {system_throughput:.2f} ток/сек")
print(f"Средняя скорость на одного пользователя: {system_throughput / 15:.2f} ток/сек")

# --- сборка итоговой таблицы ---
results = {
    "Метрика":[
        "Потребление VRAM (Веса)", 
        "Скорость генерации (1 клиент)", 
        "Inter-Token Latency (ITL)", 
        "Размер батча (Concurrency)", 
        "Общий Throughput системы"
    ],
    "Значение":[
        "~8.5 ГБ (AWQ)", 
        f"{single_speed:.2f} ток/сек", 
        f"{itl_ms:.1f} мс", 
        "15", 
        f"{system_throughput:.2f} ток/сек"
    ]
}

df = pd.DataFrame(results)
print("\n=== ИТОГОВАЯ ТАБЛИЦА ===")
print(df.to_markdown(index=False))

2026-03-05 20:00:41.515757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772740841.740797      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772740841.802966      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772740842.312082      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772740842.312115      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772740842.312118      55 computation_placer.cc:177] computation placer alr

Инициализация vLLM Engine...
INFO 03-05 20:01:07 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'attention_backend': 'TRITON_ATTN', 'model': 'casperhansen/deepseek-r1-distill-qwen-14b-awq'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json: 0.00B [00:00, ?B/s]

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Метрики для одного пользователя:
Скорость генерации: 11.53 ток/сек
Задержка между токенами (ITL): 86.70 мс

Запуск нагрузочного теста (15 одновременных запросов)...


Adding requests:   0%|          | 0/15 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Метрики под нагрузкой (Батч = 15):
Всего сгенерировано токенов: 4593
Общее время генерации: 36.45 сек
Общая пропускная способность (Throughput): 125.99 ток/сек
Средняя скорость на одного пользователя: 8.40 ток/сек

=== ИТОГОВАЯ ТАБЛИЦА ===
| Метрика                       | Значение       |
|:------------------------------|:---------------|
| Потребление VRAM (Веса)       | ~8.5 ГБ (AWQ)  |
| Скорость генерации (1 клиент) | 11.53 ток/сек  |
| Inter-Token Latency (ITL)     | 86.7 мс        |
| Размер батча (Concurrency)    | 15             |
| Общий Throughput системы      | 125.99 ток/сек |
